In [70]:
import pandas as pd

top_pairs = pd.read_csv("top_1000_human_mouse_cui_pairs.csv")
top_human_within = pd.read_csv("top_100_within_human_cui_pairs.csv")
top_mouse_within = pd.read_csv("top_100_within_mouse_cui_pairs.csv")
all_top_pairs = pd.read_csv("top_all_cui_similarity_pairs.csv")

human_preds = pd.read_csv("results/top1/human_title_summary_preds.csv")
mouse_preds = pd.read_csv("results/top1/mouse_title_summary_preds.csv")

import json
from pathlib import Path

# metadata human
path = Path("metadata/metadata_human.json")

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for gse_id, meta in data.items():
    row = {"gse_id": gse_id}
    row.update(meta)
    rows.append(row)

metadata_human = pd.DataFrame(rows)

# metadata_human.head()

# metadata mouse
path = Path("metadata/metadata_mouse.json")

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for gse_id, meta in data.items():
    row = {"gse_id": gse_id}
    row.update(meta)
    rows.append(row)

metadata_mouse = pd.DataFrame(rows)

# metadata_mouse.head()

In [71]:
# exclude pairs with exactly the same CUI set

n_before = len(top_pairs)

exact_same_pairs = top_pairs[
    (top_pairs["jaccard"] == 1.0) &
    (top_pairs["containment"] == 1.0)
]

print(f"Exact same CUI-set pairs found: {len(exact_same_pairs)}")

# optional: inspect them first
display(exact_same_pairs[[
    "human_gse", "mouse_gse"
    # ,
    # "jaccard", "containment", "overlap_count",
    # "human_title", "mouse_title"
]].head(30))

top_pairs = top_pairs[
    ~(
        (top_pairs["jaccard"] == 1.0) &
        (top_pairs["containment"] == 1.0)
    )
].reset_index(drop=True)

n_after = len(top_pairs)

print(f"Removed {n_before - n_after} exact same pairs.")
print(f"Remaining pairs: {n_after}")

# top_pairs.to_csv(output_path, index=False)
# print(f"Saved filtered file to: {output_path}")

Exact same CUI-set pairs found: 12


,human_gse,mouse_gse
0,GSE71511,GSE71513
1,GSE87042,GSE87043
2,GSE103658,GSE103725
3,GSE77702,GSE77703
4,GSE77704,GSE77703
5,GSE81148,GSE81149
6,GSE102902,GSE102903
7,GSE95516,GSE85627
8,GSE105137,GSE105138
9,GSE81328,GSE81329


Removed 12 exact same pairs.
Remaining pairs: 988


In [72]:
# remove pairs where human summary and mouse summary are exactly the same

top_pairs["human_summary"] = top_pairs["human_gse"].map(
    metadata_human.set_index("gse_id")["Summary"]
)

top_pairs["mouse_summary"] = top_pairs["mouse_gse"].map(
    metadata_mouse.set_index("gse_id")["Summary"]
)

same_summary_pairs = top_pairs[
    top_pairs["human_summary"] == top_pairs["mouse_summary"]
]

print(f"Pairs with identical summaries: {len(same_summary_pairs)}")

same_summary_pairs.to_csv("same_summary_pairs.csv")

display(same_summary_pairs[[
    "human_gse", "mouse_gse"
    # ,
    # "jaccard", "containment", "overlap_count"
    # , "human_title", "mouse_title"
    , "human_summary", "mouse_summary"
]])

same_title_pairs = top_pairs[
    top_pairs["human_title"] == top_pairs["mouse_title"]
]

print(f"Pairs with identical titles: {len(same_title_pairs)}")

display(same_title_pairs[[
    "human_gse", "mouse_gse",
    "jaccard", "containment", "overlap_count",
    "human_title", "mouse_title"
]])

top_pairs = top_pairs[
    top_pairs["human_summary"] != top_pairs["mouse_summary"]
].reset_index(drop=True)

# top_pairs.to_csv(output_path, index=False)

Pairs with identical summaries: 30


,human_gse,mouse_gse,human_summary,mouse_summary
0,GSE103630,GSE103725,Treatment of advanced V600BRAF mutant melanoma...,Treatment of advanced V600BRAF mutant melanoma...
1,GSE57096,GSE49622,Adult germline stem cells (AGSCs) are multifun...,Adult germline stem cells (AGSCs) are multifun...
2,GSE96975,GSE96991,Cardiac fibrosis is the final common pathology...,Cardiac fibrosis is the final common pathology...
3,GSE97358,GSE96991,Cardiac fibrosis is the final common pathology...,Cardiac fibrosis is the final common pathology...
4,GSE75299,GSE103725,Treatment of advanced V600BRAF mutant melanoma...,Treatment of advanced V600BRAF mutant melanoma...
5,GSE97053,GSE97054,The lung alveolus is the primary site of gas e...,The lung alveolus is the primary site of gas e...
6,GSE83652,GSE83651,Half of prostate cancers are caused by a gene-...,Half of prostate cancers are caused by a gene-...
7,GSE103630,GSE103713,Treatment of advanced V600BRAF mutant melanoma...,Treatment of advanced V600BRAF mutant melanoma...
8,GSE103687,GSE103725,Treatment of advanced V600BRAF mutant melanoma...,Treatment of advanced V600BRAF mutant melanoma...
9,GSE103688,GSE103725,Treatment of advanced V600BRAF mutant melanoma...,Treatment of advanced V600BRAF mutant melanoma...


Pairs with identical titles: 0


,human_gse,mouse_gse,jaccard,containment,overlap_count,human_title,mouse_title


In [41]:
print(human_preds[human_preds['ID'] == 'GSE83492'])
print(mouse_preds[mouse_preds['ID'] == 'GSE83991'])

            ID       mondo_id      prob  log2(prob/prior) related_words
2486  GSE83492  MONDO_0005061  0.458925          4.678117          lung
            ID       mondo_id      prob  log2(prob/prior) related_words
2953  GSE83991  MONDO_0005061  0.332263          4.212185          lung


In [69]:
# top_pairs.head(10)

| GSE       | Organism     | Title / design                           | Pair index | Pair index (s) |
| --------- | ------------ | ---------------------------------------- | ---------- | -------------- |
| GSE75299  | Homo sapiens | patient melanoma RNA-seq                 | 4, 19, 23  | |
| GSE103658 | Homo sapiens | patient batch3                           | 11, 17     | |
| GSE103630 | Homo sapiens | melanoma cell lines pre/post resistance  | 0, 7, 14   | |
| GSE103712 | Mus musculus | CD45neg YUMM1.7 mouse melanoma           | 14, 17, 23 | 20, 21, 29 |
| GSE103713 | Mus musculus | YUMM1.7 PD-L2 overexpression in NSG mice | 7, 11, 19  | 15, 16, 26 |
| GSE103725 | Mus musculus | YUMM1.7 mouse melanoma MAPKi treatment   | 0, 4       | 8, 9, 27 |

- same project-level abstract
- different SubSeries & different organism / sample system / experimental design